# DTLN 256-Unit Model GPU Training Notebook (Google Colab)
This self-contained notebook trains the **256-unit DTLN model** (`numUnits=256`, `numLayer=1`, **1,446,145 parameters**) on the **4,200-pair dataset** using GPU acceleration.

### Manual Steps Before Running:
1. Upload `data_full.zip` from your local machine to your **Google Drive** in a folder named `DTLN` (i.e., `My Drive/DTLN/data_full.zip`).
2. Select GPU Hardware Accelerator in Colab: **Runtime > Change runtime type > T4 GPU (or A100)**.
3. Run the cells sequentially below.

## Cell 1: GPU Inspection & Hardware Verification

In [ ]:
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("Physical GPUs Available:", len(gpus))
if len(gpus) > 0:
    print(f"SUCCESS: Execution Device: GPU ({gpus[0].name})")
else:
    print("\n" + "="*80)
    print(" WARNING: NO GPU DETECTED!")
    print(" Please enable GPU acceleration: Runtime > Change runtime type > T4 GPU")
    print("="*80 + "\n")

## Cell 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3: Install Required Audio Dependencies

In [ ]:
!pip install -q soundfile wavinfo

## Cell 4: Unzip Dataset Archive (`data_full.zip`) to Colab Disk

In [ ]:
import os
import zipfile

drive_zip_path = "/content/drive/MyDrive/DTLN/data_full.zip"
target_dir = "/content/data_full"

if not os.path.exists(drive_zip_path):
    raise FileNotFoundError(f"Dataset zip file not found at '{drive_zip_path}'. Please upload 'data_full.zip' to Google Drive under folder 'DTLN'.")

print(f"Unzipping '{drive_zip_path}' to '/content'...")
with zipfile.ZipFile(drive_zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content")

print("SUCCESS: Unzip complete! Verifying contents:")
print("  train_mix:   ", len(os.listdir('/content/data_full/train_mix')), "files")
print("  train_speech:", len(os.listdir('/content/data_full/train_speech')), "files")
print("  val_mix:     ", len(os.listdir('/content/data_full/val_mix')), "files")
print("  val_speech:  ", len(os.listdir('/content/data_full/val_speech')), "files")

## Cell 5: Self-Contained DTLN Architecture & Dataset Generator

In [ ]:
import os, fnmatch
import tensorflow.keras as keras
import keras.ops as ops
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Activation, Dense, LSTM, Dropout, Lambda, Input, Multiply, Layer, Conv1D
from tensorflow.keras.callbacks import ReduceLROnPlateau, CSVLogger, EarlyStopping, ModelCheckpoint
import tensorflow as tf
import soundfile as sf
from wavinfo import WavInfoReader
from random import shuffle, seed
import numpy as np

class audio_generator():
    def __init__(self, path_to_input, path_to_s1, len_of_samples, fs, train_flag=False):
        self.path_to_input = path_to_input
        self.path_to_s1 = path_to_s1
        self.len_of_samples = len_of_samples
        self.fs = fs
        self.train_flag = train_flag
        self.count_samples()
        self.create_tf_data_obj()

    def count_samples(self):
        self.file_names = fnmatch.filter(os.listdir(self.path_to_input), '*.wav')
        self.total_samples = 0
        for file in self.file_names:
            info = WavInfoReader(os.path.join(self.path_to_input, file))
            self.total_samples += int(np.fix(info.data.frame_count / self.len_of_samples))

    def create_generator(self):
        if self.train_flag:
            shuffle(self.file_names)
        for file in self.file_names:
            noisy, fs_1 = sf.read(os.path.join(self.path_to_input, file))
            speech, fs_2 = sf.read(os.path.join(self.path_to_s1, file))
            if fs_1 != self.fs or fs_2 != self.fs:
                raise ValueError('Sampling rates do not match.')
            if noisy.ndim != 1 or speech.ndim != 1:
                raise ValueError('Audio must be single channel.')
            num_samples = int(np.fix(noisy.shape[0] / self.len_of_samples))
            for idx in range(num_samples):
                in_dat = noisy[int(idx * self.len_of_samples):int((idx + 1) * self.len_of_samples)]
                tar_dat = speech[int(idx * self.len_of_samples):int((idx + 1) * self.len_of_samples)]
                yield in_dat.astype('float32'), tar_dat.astype('float32')

    def create_tf_data_obj(self):
        self.tf_data_set = tf.data.Dataset.from_generator(
            self.create_generator,
            (tf.float32, tf.float32),
            output_shapes=(tf.TensorShape([self.len_of_samples]), tf.TensorShape([self.len_of_samples])),
            args=None
        )

class InstantLayerNormalization(Layer):
    def __init__(self, **kwargs):
        super(InstantLayerNormalization, self).__init__(**kwargs)
        self.epsilon = 1e-7
        self.gamma = None
        self.beta = None

    def build(self, input_shape):
        shape = input_shape[-1:]
        self.gamma = self.add_weight(shape=shape, initializer='ones', trainable=True, name='gamma')
        self.beta = self.add_weight(shape=shape, initializer='zeros', trainable=True, name='beta')

    def call(self, inputs):
        mean = tf.math.reduce_mean(inputs, axis=[-1], keepdims=True)
        variance = tf.math.reduce_mean(tf.math.square(inputs - mean), axis=[-1], keepdims=True)
        std = tf.math.sqrt(variance + self.epsilon)
        outputs = (inputs - mean) / std
        outputs = outputs * self.gamma + self.beta
        return outputs

class DTLN_model():
    def __init__(self):
        self.cost_function = self.snr_cost
        self.model = []
        self.fs = 16000
        self.batchsize = 16
        self.len_samples = 15
        self.activation = 'sigmoid'
        self.numUnits = 256
        self.numLayer = 1
        self.blockLen = 512
        self.block_shift = 128
        self.dropout = 0.25
        self.lr = 1e-3
        self.max_epochs = 80
        self.encoder_size = 256
        self.eps = 1e-7
        os.environ['PYTHONHASHSEED'] = str(42)
        seed(42)
        np.random.seed(42)
        tf.random.set_seed(42)
        physical_devices = tf.config.experimental.list_physical_devices('GPU')
        if len(physical_devices) > 0:
            for device in physical_devices:
                tf.config.experimental.set_memory_growth(device, enable=True)

    @staticmethod
    def snr_cost(s_estimate, s_true):
        snr = tf.reduce_mean(tf.math.square(s_true), axis=-1, keepdims=True) / \
            (tf.reduce_mean(tf.math.square(s_true - s_estimate), axis=-1, keepdims=True) + 1e-7)
        num = tf.math.log(snr)
        denom = tf.math.log(tf.constant(10, dtype=num.dtype))
        loss = -10 * (num / denom)
        return loss

    def lossWrapper(self):
        def lossFunction(y_true, y_pred):
            loss = tf.squeeze(self.cost_function(y_pred, y_true))
            loss = tf.reduce_mean(loss)
            return loss
        return lossFunction

    def stftLayer(self, x):
        frames = tf.signal.frame(x, self.blockLen, self.block_shift)
        stft_dat = tf.signal.rfft(frames)
        mag = tf.abs(stft_dat)
        phase = tf.math.angle(stft_dat)
        return [mag, phase]

    def ifftLayer(self, x):
        s1_stft = (tf.cast(x[0], tf.complex64) * tf.exp((1j * tf.cast(x[1], tf.complex64))))
        return tf.signal.irfft(s1_stft)

    def overlapAddLayer(self, x):
        return tf.signal.overlap_and_add(x, self.block_shift)

    def seperation_kernel(self, num_layer, mask_size, x, stateful=False):
        for idx in range(num_layer):
            x = LSTM(self.numUnits, return_sequences=True, stateful=stateful)(x)
            if idx < (num_layer - 1):
                x = Dropout(self.dropout)(x)
        mask = Dense(mask_size)(x)
        mask = Activation(self.activation)(mask)
        return mask

    def build_DTLN_model(self, norm_stft=False):
        time_dat = Input(batch_shape=(None, None))
        mag, angle = Lambda(self.stftLayer)(time_dat)
        mag_norm = InstantLayerNormalization()(tf.math.log(mag + 1e-7)) if norm_stft else mag
        mask_1 = self.seperation_kernel(self.numLayer, (self.blockLen // 2 + 1), mag_norm)
        estimated_mag = Multiply()([mag, mask_1])
        estimated_frames_1 = Lambda(self.ifftLayer)([estimated_mag, angle])
        encoded_frames = Conv1D(self.encoder_size, 1, strides=1, use_bias=False)(estimated_frames_1)
        encoded_frames_norm = InstantLayerNormalization()(encoded_frames)
        mask_2 = self.seperation_kernel(self.numLayer, self.encoder_size, encoded_frames_norm)
        estimated = Multiply()([encoded_frames, mask_2])
        decoded_frames = Conv1D(self.blockLen, 1, padding='causal', use_bias=False)(estimated)
        estimated_sig = Lambda(self.overlapAddLayer)(decoded_frames)
        self.model = Model(inputs=time_dat, outputs=estimated_sig)
        print(self.model.summary())

    def compile_model(self):
        optimizerAdam = keras.optimizers.Adam(learning_rate=self.lr, clipnorm=3.0)
        self.model.compile(loss=self.lossWrapper(), optimizer=optimizerAdam)

    def train_model(self, runName, path_to_train_mix, path_to_train_speech, path_to_val_mix, path_to_val_speech, drive_save_dir):
        os.makedirs(drive_save_dir, exist_ok=True)
        csv_logger = CSVLogger(os.path.join(drive_save_dir, f'training_{runName}.log'))
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=10**(-10), cooldown=1)
        early_stopping = EarlyStopping(monitor='val_loss', min_delta=0, patience=10, verbose=1, mode='auto')
        weights_file_path = os.path.join(drive_save_dir, f'{runName}.weights.h5')
        checkpointer = ModelCheckpoint(weights_file_path, monitor='val_loss', verbose=1, save_best_only=True, save_weights_only=True, mode='auto', save_freq='epoch')

        len_in_samples = int(np.fix(self.fs * self.len_samples / self.block_shift) * self.block_shift)
        generator_input = audio_generator(path_to_train_mix, path_to_train_speech, len_in_samples, self.fs, train_flag=True)
        dataset = generator_input.tf_data_set.batch(self.batchsize, drop_remainder=True).repeat()
        steps_train = generator_input.total_samples // self.batchsize

        generator_val = audio_generator(path_to_val_mix, path_to_val_speech, len_in_samples, self.fs)
        dataset_val = generator_val.tf_data_set.batch(self.batchsize, drop_remainder=True).repeat()
        steps_val = generator_val.total_samples // self.batchsize

        print(f"Starting training for {self.max_epochs} max epochs...")
        self.model.fit(
            x=dataset,
            steps_per_epoch=steps_train,
            epochs=self.max_epochs,
            verbose=1,
            validation_data=dataset_val,
            validation_steps=steps_val,
            callbacks=[checkpointer, reduce_lr, csv_logger, early_stopping]
        )
        tf.keras.backend.clear_session()
        print(f"SUCCESS: Checkpoint saved persistently to Google Drive: '{weights_file_path}'")

## Cell 6: Execute Full Training Run (256-Unit Model)

In [ ]:
import time

path_train_mix = '/content/data_full/train_mix'
path_train_speech = '/content/data_full/train_speech'
path_val_mix = '/content/data_full/val_mix'
path_val_speech = '/content/data_full/val_speech'
drive_save_dir = '/content/drive/MyDrive/DTLN/models_full_run'

run_name = 'full_run_256'

model_trainer = DTLN_model()
model_trainer.numUnits = 256
model_trainer.numLayer = 1
model_trainer.max_epochs = 80
model_trainer.batchsize = 16
model_trainer.cost_function = model_trainer.snr_cost

print("=== Starting DTLN 256-Unit GPU Training ===")
print(f"  LSTM Units (`numUnits`): {model_trainer.numUnits}")
print(f"  LSTM Layers (`numLayer`): {model_trainer.numLayer}")
print(f"  Max Epoch Budget:       {model_trainer.max_epochs}")
print(f"  Batch Size:             {model_trainer.batchsize}")
print(f"  Loss Function:          {model_trainer.cost_function.__name__} (SI-SNR Loss)")
print(f"  Save Path (Drive):      {drive_save_dir}")

print("\nBuilding model...")
model_trainer.build_DTLN_model()
model_trainer.compile_model()

start_t = time.time()
model_trainer.train_model(
    run_name,
    path_train_mix,
    path_train_speech,
    path_val_mix,
    path_val_speech,
    drive_save_dir
)
elapsed = time.time() - start_t
print(f"\n=== GPU Training Completed in {elapsed / 60.0:.2f} minutes ===")

## Cell 7: Checkpoint & Log Persistence Verification

In [ ]:
import os

drive_weights = '/content/drive/MyDrive/DTLN/models_full_run/full_run_256.weights.h5'
drive_log = '/content/drive/MyDrive/DTLN/models_full_run/training_full_run_256.log'

print("=== Google Drive Checkpoint Verification ===")
if os.path.exists(drive_weights):
    print(f"SUCCESS: Trained Weights file found at: '{drive_weights}' ({os.path.getsize(drive_weights)} bytes)")
else:
    print(f"WARNING: Weights file not found at '{drive_weights}'!")

if os.path.exists(drive_log):
    print(f"SUCCESS: Training log file found at: '{drive_log}'")
    print("\nLast 10 Epoch Log entries:")
    with open(drive_log, 'r') as f:
        lines = f.readlines()
        for line in lines[-10:]:
            print("  ", line.strip())
else:
    print(f"WARNING: Training log file not found at '{drive_log}'!")